# Learning a sampling policy with PO-MPC

Policy Optimization–MPC alternates model-based planning with off-policy learning. In this example a point mass must brake near the origin. The planner predicts the outcome of candidate acceleration sequences, estimates their future return, then fits a Gaussian to the best sequences. The first-action distribution becomes a training target for an adaptive prior. That prior regularizes a learned policy that supplies a fraction of the candidates at the next planning step. [Serra-Gómez et al., ICML 2026, Algorithm 2](https://proceedings.mlr.press/v306/serra-gomez26a.html)

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All rights reserved.  
Authors: Frank Dellaert and Akash Ratheesh. See LICENSE for details.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam_unstable/planning/doc/PolicyOptimizationMPC.ipynb)

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import gtsam_unstable as gu
import plotly.graph_objects as go
from IPython.display import display

## Configure exploration and learning

At each of three planning iterations, 12 candidates sample actions from the actor at their own predicted states. The remaining 36 sample entire sequences from the current trajectory distribution. PO-MPC ranks their discounted rewards, adds a terminal bootstrap value and fits the best eight trajectories with exponential weights. A smaller positive temperature concentrates weight on the highest returns.

The C++ implementation uses fully observed vector states and small CPU neural networks. An identity encoder supplies state features to the learned transition, reward, policy, prior and categorical critics. The scalar critic support should cover the task's discounted reward scale.

In [3]:
params = gu.PolicyOptimizationMPCParams()
params.horizon = 4
params.samples, params.policySamples, params.elites = 48, 12, 8
params.iterations, params.hiddenSize, params.valueBins = 3, 16, 31
params.valueLimit, params.temperature, params.discount = 50.0, 0.5, 0.97
params.minStddev, params.maxStddev = 0.08, 1.0
params.batchSize, params.replayCapacity, params.warmupSteps = 8, 512, 12
params.updateInterval, params.reanalyzeInterval = 1, 8
params.klWeight, params.learningRate, params.seed = 0.2, 0.01, 42
params.learnedPrior, params.forwardPriorKL = True, False

controller = gu.PolicyOptimizationMPC(2, 1, params)

## Define the measured reward and the known planning cost

A state contains position and velocity. The environment returns a tracking reward measured after each simulated action. PO-MPC learns this reward and the state increment from recorded transitions. A separate graph specifies a known control-effort penalty on each action. Its factor errors enter the planning score once per trajectory. Dynamic vector priors on keys `U(k)` match the control vectors stored by the C++ rollout.

In [4]:
dt = 0.15

def environment_step(state, action):
    position, velocity = state
    acceleration = float(action[0]) - 0.3 * velocity
    next_state = np.array([
        position + dt * velocity + 0.5 * dt**2 * acceleration,
        velocity + dt * acceleration,
    ])
    reward = -float(next_state[0]**2 + 0.15 * next_state[1]**2)
    return next_state, reward

effort_cost = gtsam.NonlinearFactorGraph()
for k in range(params.horizon):
    effort_cost.add(gtsam.PriorFactorVector(
        gtsam.symbol("u", k), np.zeros(1),
        gtsam.noiseModel.Isotropic.Sigma(1, 3.0)))

## Collect transitions and train from replay

During warmup the actor explores and the replay buffer stores broad Gaussian planning targets. When warmup ends the model and bootstrap critic pretrain on replay. Subsequent calls to `observe` update both critics, the actor, and the intermediate prior at the configured interval. Every eighth update refreshes planning targets on a sampled trajectory using the current model.

Use `observe(..., terminal=True)` at a terminal state. At a time limit, record the transition with `terminal=False` and call `reset()` before starting the next episode. Both end the replay sequence and clear the warm start. Only the time-limited transition bootstraps from its recorded next state.

In [5]:
episode_rewards, first_action_spreads = [], []
initial_actor = controller.policyMean(np.array([1.0, 0.0])).copy()
steps = 24
for episode in range(8):
    state = np.array([(-1.0)**episode, 0.0])
    episode_return = 0.0
    for k in range(steps):
        plan = controller.plan(state, effort_cost)
        next_state, reward = environment_step(state, plan.action)
        controller.observe(state, plan, reward, next_state, k == steps - 1)
        first_action_spreads.append(float(plan.stddev[0, 0]))
        state = next_state
        episode_return += reward
    episode_rewards.append(episode_return)

trained_actor = controller.policyMean(np.array([1.0, 0.0]))
assert controller.replaySize() == steps * len(episode_rewards)
assert np.all(np.isfinite(episode_rewards)) and np.all(np.isfinite(trained_actor))
print("Transitions in replay", controller.replaySize())
print("Actor mean at position 1 before and after training", initial_actor, trained_actor)

Transitions in replay 192
Actor mean at position 1 before and after training [0.00656909] [-0.10538966]


## Inspect the planner and sampling policy

Episode reward measures the executed closed-loop trajectory, while the fitted spread describes the planner's uncertainty across elite trajectories at each decision. A short seeded run illustrates how to collect and inspect both quantities. The return can vary as the model, policy, prior and sampled action all change. The interactive chart loads its version-pinned Plotly runtime from the CDN and requires an internet connection.

In [6]:
figure = go.Figure(go.Scatter(
    x=list(range(1, len(episode_rewards) + 1)),
    y=episode_rewards, mode="lines+markers", name="Episode reward"))
figure.update_layout(xaxis_title="Episode", yaxis_title="Sum of measured rewards")
from html import escape

document = figure.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display({'text/html': f'<iframe title="Learning a sampling policy with PO-MPC" style="width:100%;border:0" '
             f'height="{figure.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'}, raw=True)
print("Range of fitted first-action standard deviations",
      np.min(first_action_spreads), np.max(first_action_spreads))

<iframe title="Learning a sampling policy with PO-MPC" style="width:100%;border:0" height="480" srcdoc="<html>
<head><meta charset="utf-8" /></head>
<body>
 <div> <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.1.1.min.js" integrity="sha256-HUEFyfiTnZJxCxur99FjbKYTvKSzwDaD3/x5TqHpFu4=" crossorigin="anonymous"></script> <div id="85fb43db-23e4-403c-8296-e17a7ea8eaf3" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script type="text/javascript"> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("85fb43db-23e4-403c-8296-e17a7ea8eaf3")) { Plotly.newPlot( "85fb43db-23e4-403c-8296-e17a7ea8eaf3", [{"mode":"lines+markers","name":"Episode reward","x":[1,2,3,4,5,6,7,8],"y":[-19.97051939849097,-37.439478018072364,-71.555985118032,-42.65131066135016,-37.94618427875825,-60.90557645752048,-8.804988559680613,-31.676939575248912],"type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"parcoords":[{"type":"parcoords","line":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolargl":[{"type":"scatterpolargl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"#E5ECF6","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"bar"}],"scattergeo":[{"type":"scattergeo","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolar":[{"type":"scatterpolar","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"histogram":[{"marker":{"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"histogram"}],"scattergl":[{"type":"scattergl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatter3d":[{"type":"scatter3d","line":{"colorbar":{"outlinew

Range of fitted first-action standard deviations 0.08 1.0


With `learnedPrior=True`, the prior neural network fits the stored planning marginals. Set `forwardPriorKL=True` to fit the prior by forward KL, or leave it false to minimize reverse KL. With `learnedPrior=False`, regularization uses replay planning marginals directly. Setting `updateInterval=0` allows explicit minibatch updates with `controller.train(updates)` after observing transitions. Call `policyMean(state)` to inspect the learned actor and `plan(state, effort_cost)` to run the mixed search.